# Week 6 – Product Pricer Fine-Tuning (abdussamadbello)

Fine-tune GPT-4.1-nano on the product-pricer dataset and evaluate. Run from repo root or ensure `week6` is on `sys.path` so `pricer` can be imported.

In [ ]:
# Add week6 to path so pricer can be imported (notebook cwd = this folder)
import os
import sys
week6_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if os.path.isdir(week6_dir) and week6_dir not in sys.path:
    sys.path.insert(0, week6_dir)

import json
import builtins
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate, Tester
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
load_dotenv(override=True)
LITE_MODE = True  # set False for full dataset
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
train, val, test = Item.from_hub(dataset)
builtins.print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set")
openai = OpenAI(api_key=api_key, base_url=base_url)
builtins.print("OpenAI base_url:", base_url)

In [ ]:
fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [ ]:
def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

def make_jsonl(items):
    return "\n".join(
        '{"messages": ' + json.dumps(messages_for(item)) + '}'
        for item in items
    )

def write_jsonl(items, filename):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    Path(filename).write_text(make_jsonl(items), encoding="utf-8")

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
train_file = openai.files.create(file=Path("jsonl/fine_tune_train.jsonl"), purpose="fine-tune")
validation_file = openai.files.create(file=Path("jsonl/fine_tune_validation.jsonl"), purpose="fine-tune")
builtins.print("Train file id:", train_file.id, "Validation file id:", validation_file.id)

In [ ]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)
builtins.print("Job id:", job.id)

In [ ]:
# Poll until job is done (or check status manually)
import time
while True:
    status = openai.fine_tuning.jobs.retrieve(job.id)
    builtins.print(status.status)
    if status.status in ("succeeded", "failed", "cancelled"):
        break
    time.sleep(30)
fine_tuned_model_name = status.fine_tuned_model if status.status == "succeeded" else None

In [ ]:
def test_messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

def gpt_fine_tuned(item):
    r = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return r.choices[0].message.content

In [ ]:
if fine_tuned_model_name:
    score = evaluate(gpt_fine_tuned, test)
    builtins.print("Evaluation score:", score)
else:
    builtins.print("Job did not succeed; skip evaluation.")

## Baseline comparison & category-stratified eval

Compare **base** (non–fine-tuned) GPT-4.1-nano vs **fine-tuned** on the same test subset, then break down results by product category and add possible reasons.

In [ ]:
BASE_MODEL = "gpt-4.1-nano-2025-04-14"
EVAL_SIZE = min(200, len(test))  # cap for speed

def gpt_baseline(item):
    r = openai.chat.completions.create(
        model=BASE_MODEL,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return r.choices[0].message.content

def run_predictor(predictor, data, size):
    """Run predictor on first `size` items; return list of (item, guess, truth, error)."""
    from tqdm.notebook import tqdm
    rows = []
    for i in tqdm(range(size), desc="Predicting"):
        item = data[i]
        out = predictor(item)
        guess = Tester.post_process(out)
        truth = item.price
        error = abs(guess - truth)
        rows.append((item, guess, truth, error))
    return rows

In [ ]:
# Run both models on the same subset
builtins.print("Running baseline on", EVAL_SIZE, "test items...")
base_rows = run_predictor(gpt_baseline, test, EVAL_SIZE)
builtins.print("Running fine-tuned on", EVAL_SIZE, "test items...")
ft_rows = run_predictor(gpt_fine_tuned, test, EVAL_SIZE) if fine_tuned_model_name else []

avg_base = sum(r[3] for r in base_rows) / EVAL_SIZE
avg_ft = sum(r[3] for r in ft_rows) / EVAL_SIZE if ft_rows else None

builtins.print("Baseline (base model) mean absolute error: ${:.2f}".format(avg_base))
if avg_ft is not None:
    builtins.print("Fine-tuned mean absolute error: ${:.2f}".format(avg_ft))
    builtins.print("Improvement: ${:.2f}".format(avg_base - avg_ft))

In [ ]:
# Category-stratified: group by item.category
def stratified_stats(base_rows, ft_rows):
    from collections import defaultdict
    cat_base = defaultdict(list)  # category -> [errors]
    cat_ft = defaultdict(list)
    cat_price = defaultdict(list)
    cat_count = defaultdict(int)
    for i, (item, _, truth, err_base) in enumerate(base_rows):
        cat = item.category or "Unknown"
        cat_base[cat].append(err_base)
        cat_price[cat].append(truth)
        cat_count[cat] += 1
        if ft_rows:
            cat_ft[cat].append(ft_rows[i][3])
    rows = []
    for cat in sorted(cat_base.keys()):
        n = cat_count[cat]
        mean_price = sum(cat_price[cat]) / n
        mean_err_base = sum(cat_base[cat]) / n
        mean_err_ft = (sum(cat_ft[cat]) / n) if ft_rows else None
        rows.append({
            "category": cat,
            "n": n,
            "mean_price": mean_price,
            "mean_error_baseline": mean_err_base,
            "mean_error_finetuned": mean_err_ft,
        })
    return pd.DataFrame(rows)

df_cat = stratified_stats(base_rows, ft_rows)
df_cat

In [ ]:
# Bar chart: mean absolute error by category (baseline vs fine-tuned)
fig = go.Figure()
fig.add_trace(go.Bar(name="Baseline", x=df_cat["category"], y=df_cat["mean_error_baseline"], marker_color="steelblue"))
if "mean_error_finetuned" in df_cat.columns and df_cat["mean_error_finetuned"].notna().any():
    fig.add_trace(go.Bar(name="Fine-tuned", x=df_cat["category"], y=df_cat["mean_error_finetuned"], marker_color="darkorange"))
fig.update_layout(
    title="Mean absolute error by category",
    xaxis_title="Category",
    yaxis_title="Mean absolute error ($)",
    barmode="group",
    height=400,
)
fig.show()

### Possible reasons for per-category differences

- **Scale effect**: Categories with higher average price (e.g. Electronics) often have higher *absolute* error even if relative error is similar.
- **Sample size**: Categories with few test items have noisier metrics; small n can make a category look better or worse by chance.
- **Where fine-tuning helps**: Fine-tuning tends to help most where the base model was already uncertain or where the training data had many examples in that category.

In [ ]:
# Optional: get a short "reason" from the base model (why might this product cost ~$X?)
sample = test[0]
pred_ft = gpt_fine_tuned(sample) if fine_tuned_model_name else gpt_baseline(sample)
guess_val = Tester.post_process(pred_ft)

reason_messages = [
    {"role": "user", "content": (
        f"Product: {sample.summary[:500]}\n\n"
        f"In one short sentence, why might this product cost around ${guess_val:.0f}?"
    )}
]
try:
    r = openai.chat.completions.create(
        model=BASE_MODEL,
        messages=reason_messages,
        max_tokens=80,
    )
    builtins.print("Reason (base model):", r.choices[0].message.content)
except Exception as e:
    builtins.print("(Reason call skipped:", e, ")")